In [15]:
# ========== 导入：文本聊天 + 语音 + 工具调用所需依赖 ==========

# os：读环境变量（如 OPENAI_API_KEY）
import os
# Gradio：搭 Blocks 多控件界面（文本、麦克风、模型切换）
import gradio as gr
# json：解析 tool_call.function.arguments
import json
# 再次 import os：原代码如此保留（重复导入无害）
import os
# OpenAI 客户端：云端 GPT、语音转写/合成；也可改 base_url 接 Ollama
from openai import OpenAI
# load_dotenv：从 .env 加载密钥到进程环境
from dotenv import load_dotenv


In [2]:
# ========== 常量：两个后端的模型名集中管理 ==========

# 云端小模型：文本对话默认用它（Radio 选 OpenAI 时）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：须事先 pull；Radio 选 Ollama 时用
MODEL_LLAMA = 'llama3.2'


In [3]:
# ========== 环境初始化：双客户端 + 密钥检查 + system prompt ==========

# 云端 OpenAI 客户端（默认读环境变量里的 API Key）
openai = OpenAI()
# 本地 Ollama 的 OpenAI 兼容基址
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# 第二个客户端：同一套 SDK，指向本机 Ollama
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
# 默认模型常量（本笔记本主要靠 Radio 切换，不强制用这个）
MODEL = MODEL_LLAMA
# 加载 .env；override=True 表示用文件覆盖已有同名环境变量
load_dotenv(override=True)
# 取出 OpenAI 密钥做一次「看起来是否合法」的粗检
api_key = os.getenv('OPENAI_API_KEY')

# 粗检：存在、以 sk-proj- 开头、长度够 —— 通过就打印好消息
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    # 失败提示文案保持英文原文（依赖程序判断/课程排错笔记本约定）
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")


# system prompt：技术助手；用户明确要订酒店时必须走 book_hotel 工具
# （发给模型的指令，英文保持原样）
system_message = """
You are an expert technical assistant who helps developers debug code, explain concepts,
and solve engineering problems.

If the user asks a general technical question, respond normally.

However, if the user explicitly asks to book a hotel or make a hotel reservation,
you MUST call the book_hotel tool instead of responding conversationally.
"""


API key looks good so far


In [4]:
# ========== 酒店预订工具：模拟「真的订成功了」 ==========

def book_hotel(name: str, city: str, nights: int, start_date: str):
    # 返回确认字符串；真实项目里这里会调外部预订 API
    return f"Hotel successfully booked in {city} for {name} for {nights} nights starting {start_date}"


In [5]:
# ========== tools schema：声明给模型的可调用函数 ==========

tools = [
    {
        # OpenAI tools 条目类型固定为 function
        "type": "function",
        "function": {
            # 必须与 Python 函数名 / 分发逻辑一致
            "name": "book_hotel",
            # 描述帮助模型决定何时调用（英文保留）
            "description": "Book a hotel when a user asks to reserve or book accommodation",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "city": {"type": "string"},
                    "nights": {"type": "integer"},
                    "start_date": {"type": "string"}
                },
                # 四个参数都必填，缺一不可
                "required": ["name", "city", "nights", "start_date"]
            }
        }
    }
]


In [6]:
# ========== 模型路由：Radio 选 OpenAI 或 Ollama ==========

def get_client_and_model(choice):
    # 界面选项字符串 "OpenAI" → 云端客户端 + gpt-4o-mini
    if choice == "OpenAI":
        return openai, MODEL_GPT
    else:
        # 其他（本 UI 里是 "Ollama"）→ 本地客户端 + llama3.2
        return ollama, MODEL_LLAMA


In [ ]:
# ========== 语音转文字（STT）：麦克风文件 → 文本 ==========

def transcribe_audio(audio):
    # 没录音就返回空串，上层 voice_chat 会直接结束
    if audio is None:
        return ""

    # 以二进制打开 Gradio 给的音频 filepath
    with open(audio, "rb") as f:
        # 调用 OpenAI 转写模型（模型 id 保持原样）
        transcript = openai.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe",
            file=f
        )
    # 只要文本内容
    return transcript.text


In [8]:
# ========== 文字转语音（TTS）：回复文本 → mp3 文件 ==========

def text_to_speech(text):
    # 输出文件名（相对工作目录）；原路径字符串不改
    speech_file = "response.mp3"

    # with_streaming_response：边收边写文件，适合稍长的语音
    with openai.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    ) as response:
        # 把流式音频写到 speech_file
        response.stream_to_file(speech_file)

    # 返回路径给 Gradio Audio 控件播放
    return speech_file


In [13]:
# ========== 文本聊天（可流式 yield）+ 可选 tool calling ==========

def chat(message, history, model_choice):

    # Gradio 有时传入 None：统一成空列表
    history = history or []

    # 先把本轮用户消息写入 history（messages 格式）
    history.append({"role":"user","content":message})

    # 按 Radio 选择拿到 client 与 model 名
    client, model = get_client_and_model(model_choice)


    # 文本路径用的 system 指令（英文保留）；再拼上完整 history
    messages = [
        {"role":"system",
        "content":"You are a general technical assistant. Only call the hotel booking tool if the request is about booking a hotel."}
    ] + history

    # 非流式先问一轮：让模型决定要不要 tool_calls（tool_choice=auto）
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    # 取出助手消息对象（可能带 tool_calls）
    msg = response.choices[0].message

    # 若模型请求调用工具
    if msg.tool_calls:

        for tool_call in msg.tool_calls:

            # 只处理 book_hotel；其他名字忽略
            if tool_call.function.name == "book_hotel":

                # 参数 JSON → dict
                args = json.loads(tool_call.function.arguments)

                # 调用本地模拟预订函数
                result = book_hotel(
                    args["name"],
                    args["city"],
                    args["nights"],
                    args["start_date"]
                )

                # 先追加助手那条（含 tool_calls）
                messages.append(msg)

                # 再追加 tool 结果，带上 tool_call_id
                messages.append({
                    "role":"tool",
                    "tool_call_id":tool_call.id,
                    "content":result
                })

        # 工具跑完后：再流式生成最终自然语言回复
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )

    else:

        # 无工具：直接对流式再生成一遍（与上面同一套 messages）
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )

    # 累积流式文本
    partial = ""

    # 先放一条空的 assistant，后面边收边改 content
    history.append({"role":"assistant","content":""})

    for chunk in stream:
        # 有的 chunk 只有 role/空 delta，要判 content 是否存在
        if chunk.choices[0].delta.content:
            partial += chunk.choices[0].delta.content
            history[-1]["content"] = partial
            # yield 整份 history：Gradio Chatbot 才能「打字机」更新
            yield history


In [10]:
# ========== 语音聊天：STT →（可选工具）→ 文本回复 → TTS ==========

def voice_chat(audio, history, model_choice):

    # 麦克风音频 → 用户文本
    user_text = transcribe_audio(audio)

    # 转写失败/空录音：原样返回 history，音频输出清空
    if not user_text:
        return history, None

    # 用户文本写入对话历史
    history.append({"role":"user","content":user_text})

    # 按界面选择路由客户端与模型
    client, model = get_client_and_model(model_choice)

    # 语音路径用完整 system_message（含「必须 call book_hotel」）
    messages = [{"role":"system","content":system_message}] + history

    # 注意：原代码 tools=book_hotel（函数对象），与文本路径的 tools 列表不同——逻辑保持原样
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=book_hotel,
        tool_choice="auto"
    )

    msg = response.choices[0].message

    if msg.tool_calls:

        # 本实现只取第一个 tool_call
        tool_call = msg.tool_calls[0]
        args = json.loads(tool_call.function.arguments)

        # **args 按名字解包传给 book_hotel
        result = book_hotel(**args)

        # 把助手消息与 tool 结果追加进 history（后续再拼 system+history）
        history.append(msg)

        history.append({
            "role":"tool",
            "tool_call_id": tool_call.id,
            "content":result
        })

        # 带着工具结果再请求一次最终文本回复（非流式）
        final = client.chat.completions.create(
            model=model,
            messages=[{"role":"system","content":system_message}] + history
        )

        reply = final.choices[0].message.content

    else:
        # 无工具：直接用首轮助手文本
        reply = msg.content

    # 助手回复写入历史
    history.append({"role":"assistant","content":reply})

    # 文本 → 语音文件，供界面自动播放
    audio_reply = text_to_speech(reply)

    return history, audio_reply


In [11]:
# ========== 清空：聊天记录 + 输入/输出音频 ==========

def clear_all():
    # 返回三个 None/空列表，对应 chatbot、audio_input、audio_output
    return [], None, None


In [16]:
# ========== Gradio Blocks：文本 + 语音 + 模型切换 UI ==========

with gr.Blocks() as demo:

    # 标题（界面展示字符串保持原样）
    gr.Markdown("# 🎧 Technical Assistant with Voice + Tool Use")

    # 单选：云端 OpenAI 或本地 Ollama
    model_choice = gr.Radio(["OpenAI","Ollama"], value="OpenAI")

    # 消息格式聊天窗口
    chatbot = gr.Chatbot(type="messages")

    # 文本输入框 + 发送按钮
    msg = gr.Textbox(placeholder="Ask a technical question...")
    send = gr.Button("Send")

    # 麦克风录音（filepath）+ 语音提问按钮
    audio_input = gr.Audio(sources=["microphone"], type="filepath")
    voice_btn = gr.Button("Ask with Voice")

    # 助手语音回复；autoplay 自动播放
    audio_output = gr.Audio(autoplay=True)

    clear_btn = gr.Button("Clear Chat")

    # 回车提交文本：跑 chat，再清空输入框
    msg.submit(
        chat,
        inputs=[msg, chatbot, model_choice],
        outputs=chatbot
    ).then(lambda: "", outputs=msg)

    # 点击 Send：同上
    send.click(
        chat,
        inputs=[msg, chatbot, model_choice],
        outputs=chatbot
    ).then(lambda: "", outputs=msg)

    # 语音按钮：voice_chat 更新 chatbot + 播放音频，再清空麦克风控件
    voice_btn.click(
        voice_chat,
        inputs=[audio_input, chatbot, model_choice],
        outputs=[chatbot, audio_output]
    ).then(lambda: None, outputs=audio_input)

    # 清空聊天与音频控件
    clear_btn.click(
        clear_all,
        outputs=[chatbot, audio_input, audio_output]
    )

# share=True 生成临时公网链接；inbrowser=True 尝试自动打开浏览器
demo.launch(share=True, inbrowser=True)


* Running on local URL:  http://127.0.0.1:7870
* Running on public URL: https://0eecd474e4ff05230b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
